<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session10_fine_tuning_with_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۰: فاین‌تیون مدل‌های زبانی با LoRA و QLoRA

### اهداف این جلسه

- تشخیص اینکه چه زمانی فاین‌تیون لازم است و چه زمانی پرامپت یا RAG کافی است
- آماده‌سازی دادهٔ دستورالعملی در قالب گفتگو و آشنایی با Chat Template
- درک ریاضی LoRA و صرفه‌جویی آن در تعداد پارامترهای قابل آموزش
- آموزش یک مدل ۱٫۵ میلیارد پارامتری با QLoRA روی GPU رایگان Colab با TRL
- مقایسهٔ مدل فاین‌تیون‌شده با مدل پایه از نظر دقت، طول پرامپت و سرعت

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. فاین‌تیون چه زمانی؟

**فاین‌تیون (Fine-tuning)** یعنی ادامهٔ آموزش یک مدل ازپیش‌آموزش‌دیده روی دادهٔ خودمان، تا رفتارش برای یک کار مشخص تغییر کند. در جلسهٔ ۱۴ دورهٔ یادگیری عمیق ParsBERT را برای تحلیل احساس فاین‌تیون کردیم. اینجا همین کار را برای یک مدل زبانی مولد انجام می‌دهیم.

پیش از فاین‌تیون این ترتیب را امتحان کنید:

| راه | مناسب برای | هزینه |
|---|---|---|
| پرامپت بهتر و Few-shot | بیشتر کارها | تقریباً صفر |
| RAG (جلسه‌های ۶ تا ۹) | دانش و واقعیت‌هایی که تغییر می‌کنند | کم |
| فاین‌تیون | قالب خروجی ثابت، سبک و لحن، کار محدود و پرتکرار، مدل کوچک‌تر و ارزان‌تر | داده، GPU و نگهداری |

فاین‌تیون در **رفتار** خوب است، نه در افزودن قابل‌اعتماد **دانش**. مدلی که روی سیاست‌های فروشگاه فاین‌تیون شده، با تغییر سیاست‌ها باید دوباره آموزش ببیند و باز هم ممکن است جزئیات را اشتباه بگوید. برای دانش، RAG بهتر است.

### مسئلهٔ ما

در تمرین ۴ جلسهٔ ۴ از مدل خواستیم سفارش مشتری را از پیام فارسی استخراج کند، ولی برای ثبت سفارش به **شناسهٔ کالا** نیاز داریم. مدل باید یاد بگیرد «یه A55 آبی» یعنی `P1001`. با پرامپت، باید کل فهرست کالاها را در هر درخواست بفرستیم، که پرامپت را طولانی، کند و گران می‌کند. با فاین‌تیون، یک مدل **کوچک** این نگاشت و قالب خروجی را یاد می‌گیرد و پرامپت کوتاه می‌شود. این همان نوع کاری است که فاین‌تیون برایش مناسب است: محدود، پرتکرار و با خروجی ساختاریافته.

</div>

In [ ]:
!pip -q install -U transformers trl peft bitsandbytes datasets
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

import json
import re
import time

import pandas as pd
import torch
from llm_utils import download_data

def load_jsonl(name):
    with open(download_data(name), encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("orders_train.jsonl")
test_data = load_jsonl("orders_test.jsonl")
real_data = load_jsonl("orders_real.jsonl")
print(len(train_data), "train,", len(test_data), "test,", len(real_data), "real")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. داده

داده‌ها **مصنوعی**‌اند و با اسکریپت [`order_data.py`](../order_data.py) ساخته شده‌اند: برای هر کالا چند نام محاوره‌ای («گلکسی A55»، «سامسونگ A55»)، تعداد به شکل‌های مختلف («یه»، «دو تا»، «۳ عدد»)، رنگ و شهر، و ۸ الگوی جمله. ساخت دادهٔ مصنوعی با الگو، یا با یک مدل بزرگ‌تر، روش رایجی برای شروع است، ولی همیشه باید روی **دادهٔ واقعی** هم ارزیابی شود.

- `orders_train.jsonl`: ۱۲۰۰ نمونهٔ آموزشی
- `orders_test.jsonl`: ۱۵۰ نمونه با همان الگوها و ۵۰ نمونه با **دو الگوی جدید** که در آموزش نبوده‌اند
- `orders_real.jsonl`: ۱۲ پیام دست‌نویس با سبک‌هایی که اسکریپت تولید نمی‌کند

</div>

In [ ]:
for example in train_data[:3] + real_data[:2]:
    print(example["message"], "\n   ->", example["order"])
test_split = ["seen templates"] * 150 + ["new templates"] * 50

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. مدل پایه و ارزیابی پیش از آموزش

مدل `Qwen2.5-1.5B-Instruct` را انتخاب می‌کنیم: به اندازهٔ کافی کوچک است که روی GPU رایگان Colab آموزش ببیند و فارسی را تا حدی می‌فهمد. مدل را با **کوانتیزه‌سازی ۴ بیتی** بارگذاری می‌کنیم؛ دلیلش را در بخش ۵ می‌بینیم.

</div>

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"          # for batched generation
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
print(f"GPU memory used by the model: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تابع‌های تولید و ارزیابی: پاسخ مدل باید یک JSON با چهار فیلد باشد. برای هر نمونه بررسی می‌کنیم JSON معتبر است و هر فیلد درست است. معیار اصلی **دقت کامل** است: هر چهار فیلد درست باشند، چون سفارشی که فقط شهرش غلط است هم سفارش غلط است.

</div>

In [ ]:
def generate(model, conversations, batch_size=16, max_new_tokens=80):
    outputs = []
    for i in range(0, len(conversations), batch_size):
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
                 for c in conversations[i:i + batch_size]]
        batch = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        outputs += tokenizer.batch_decode(out[:, batch.input_ids.shape[1]:], skip_special_tokens=True)
    return outputs


def _same(a, b):
    norm = lambda x: None if x in (None, "", "null") else str(x).replace("\u200c", " ").strip()
    return norm(a) == norm(b)


def score(outputs, data, split=None):
    rows = []
    for out, item in zip(outputs, data):
        gold = item["order"]
        match = re.search(r"\{.*\}", out, re.S)
        try:
            pred = json.loads(match.group()) if match else None
        except json.JSONDecodeError:
            pred = None
        row = {"valid_json": isinstance(pred, dict)}
        for field in ["product_id", "quantity", "color", "city"]:
            row[field] = row["valid_json"] and _same(pred.get(field), gold[field])
        row["all_fields"] = all(row[f] for f in ["product_id", "quantity", "color", "city"])
        rows.append(row)
    df = pd.DataFrame(rows)
    if split is not None:
        df["split"] = split
    return df

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**مبنای مقایسه:** مدل پایه با پرامپتی که کل فهرست کالاها را دارد.

</div>

In [ ]:
products = pd.read_csv(download_data("products.csv"))
catalog = "\n".join(f"{r.product_id}: {r.name}" for r in products.itertuples())

SYSTEM_WITH_CATALOG = f"""Extract the order from the Persian customer message.
Return only JSON with keys product_id, quantity, color, city.
- product_id: the matching id from the catalog below.
- quantity: an integer; 1 if not mentioned.
- color and city: in Persian, or null if not mentioned.

Catalog:
{catalog}"""


def conversations(data, system):
    return [[{"role": "system", "content": system}, {"role": "user", "content": d["message"]}] for d in data]


start = time.time()
base_outputs = generate(model, conversations(test_data, SYSTEM_WITH_CATALOG))
base_seconds = time.time() - start
base_scores = score(base_outputs, test_data, test_split)
print(f"time: {base_seconds:.0f}s")
base_scores.groupby("split").mean().round(3)

In [ ]:
for out, item in list(zip(base_outputs, test_data))[:3]:
    print(item["message"], "\n  model:", out.strip(), "\n  gold: ", json.dumps(item["order"], ensure_ascii=False), "\n")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مدل ۱٫۵ میلیاردی با پرامپت طولانی معمولاً قالب JSON را رعایت می‌کند، ولی در نگاشت نام محاوره‌ای به شناسه، تشخیص تعداد و جدا کردن «رنگ» از نام کالا خطا دارد.

## ۴. LoRA: آموزش با کسری از پارامترها

فاین‌تیون کامل یعنی به‌روزرسانی همهٔ ۱٫۵ میلیارد پارامتر. برای این کار باید وزن‌ها، گرادیان‌ها و حالت بهینه‌ساز Adam، یعنی چند برابر حجم مدل، در حافظهٔ GPU جا شوند. **LoRA (Low-Rank Adaptation)** وزن‌های اصلی را ثابت نگه می‌دارد و فقط یک «اصلاحیهٔ» کوچک یاد می‌گیرد:

$$W' = W + \Delta W = W + \frac{\alpha}{r} B A, \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d, k)$$

ایده: تغییری که فاین‌تیون در وزن‌ها لازم دارد، **رتبهٔ پایینی** دارد و با ضرب دو ماتریس باریک قابل تقریب است. ماتریس $B$ با صفر مقداردهی می‌شود، پس در شروع آموزش $\Delta W = 0$ است و مدل دقیقاً همان مدل پایه است.

برای یک لایهٔ خطی $1536 \times 1536$:

</div>

In [ ]:
d = k = 1536
for r in [8, 16, 64]:
    full, lora = d * k, r * (d + k)
    print(f"r={r:3d}: LoRA parameters {lora:>9,} = {100 * lora / full:.2f}% of the full matrix ({full:,})")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

| ابرپارامتر | معنی | مقدار رایج |
|---|---|---|
| `r` | رتبهٔ اصلاحیه؛ ظرفیت یادگیری | ۸ تا ۶۴ |
| `lora_alpha` | ضریب مقیاس؛ معمولاً دو برابر `r` | ۱۶ تا ۱۲۸ |
| `target_modules` | لایه‌هایی که LoRA می‌گیرند | همهٔ لایه‌های خطی (`all-linear`) |
| `lora_dropout` | منظم‌سازی | ۰ تا ۰٫۱ |

## ۵. QLoRA: مدل ۴ بیتی + LoRA

**QLoRA** یک قدم جلوتر می‌رود: وزن‌های ثابت مدل پایه را با کوانتیزه‌سازی **NF4** در ۴ بیت نگه می‌دارد و فقط ماتریس‌های LoRA با دقت ۱۶ یا ۳۲ بیت آموزش می‌بینند. حجم مدل ۱٫۵ میلیاردی از حدود ۳ گیگابایت در fp16 به حدود ۱ گیگابایت می‌رسد. با QLoRA حتی مدل‌های ۷ تا ۸ میلیاردی روی GPU رایگان T4 با ۱۵ گیگابایت حافظه قابل فاین‌تیون‌اند.

## ۶. آماده‌سازی داده برای آموزش

داده را در قالب **Prompt-Completion** گفتگومحور آماده می‌کنیم: `prompt` شامل پیام system و پیام کاربر است و `completion` پاسخ درست دستیار. کتابخانهٔ TRL در این قالب، **loss را فقط روی بخش completion** حساب می‌کند. مدل نباید تولید پیام کاربر را یاد بگیرد، فقط پاسخ دادن را.

پیام system این بار **کوتاه** است و فهرست کالا ندارد. مدل باید نگاشت را از داده یاد بگیرد.

</div>

In [ ]:
from datasets import Dataset

SYSTEM_SHORT = ("Extract the order from the Persian customer message as JSON with keys "
                "product_id, quantity, color, city.")


def to_example(item):
    return {"prompt": [{"role": "system", "content": SYSTEM_SHORT},
                       {"role": "user", "content": item["message"]}],
            "completion": [{"role": "assistant",
                            "content": json.dumps(item["order"], ensure_ascii=False)}]}


train_dataset = Dataset.from_list([to_example(d) for d in train_data])
print(train_dataset[0])
print("\nas the model sees it:\n" + tokenizer.apply_chat_template(
    train_dataset[0]["prompt"] + train_dataset[0]["completion"], tokenize=False))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۷. آموزش با TRL

`SFTTrainer` از کتابخانهٔ TRL، **Supervised Fine-Tuning**، کارهای تکراری را انجام می‌دهد: اعمال Chat Template، توکن‌سازی، ماسک کردن بخش prompt در loss، افزودن LoRA و آماده‌سازی مدل ۴ بیتی برای آموزش. حلقهٔ آموزش همان حلقه‌ای است که در جلسهٔ ۴ دورهٔ یادگیری عمیق نوشتیم.

GPU مدل T4 از `bfloat16` پشتیبانی نمی‌کند، پس `fp16=True` می‌گذاریم. آموزش حدود ۵ تا ۱۰ دقیقه طول می‌کشد.

</div>

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear",
                         task_type="CAUSAL_LM")
training_args = SFTConfig(
    output_dir="order-extractor-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,         # effective batch size 16
    learning_rate=2e-4,                    # LoRA tolerates a much larger rate than full fine-tuning
    lr_scheduler_type="cosine",
    warmup_steps=0.05,                     # a value below 1 means a fraction of all steps
    logging_steps=10,
    max_length=256,
    fp16=True,
    bf16=False,
    save_strategy="no",
    report_to="none",
)
tokenizer.padding_side = "right"           # for training
trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_dataset,
                     processing_class=tokenizer, peft_config=lora_config)
trainer.model.print_trainable_parameters()

In [ ]:
start = time.time()
trainer.train()
print(f"training time: {(time.time() - start) / 60:.1f} minutes")

In [ ]:
import matplotlib.pyplot as plt

history = pd.DataFrame([h for h in trainer.state.log_history if "loss" in h])
plt.plot(history.step, history.loss, marker="o")
plt.xlabel("step")
plt.ylabel("training loss")
plt.title("QLoRA fine-tuning")
plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۸. ارزیابی مدل فاین‌تیون‌شده

همان آزمون را با پیام system **کوتاه** اجرا می‌کنیم:

</div>

In [ ]:
tokenizer.padding_side = "left"
trainer.model.eval()
start = time.time()
tuned_outputs = generate(trainer.model, conversations(test_data, SYSTEM_SHORT))
tuned_seconds = time.time() - start
tuned_scores = score(tuned_outputs, test_data, test_split)

comparison = pd.concat({"base + catalog": base_scores.groupby("split").mean(),
                        "fine-tuned": tuned_scores.groupby("split").mean()}).round(3)
comparison

In [ ]:
prompt_tokens = {name: len(tokenizer(tokenizer.apply_chat_template(
                     conversations(test_data[:1], system)[0], tokenize=False, add_generation_prompt=True)).input_ids)
                 for name, system in [("base + catalog", SYSTEM_WITH_CATALOG), ("fine-tuned", SYSTEM_SHORT)]}
print("prompt tokens:", prompt_tokens)
print(f"generation time for 200 messages: base {base_seconds:.0f}s, fine-tuned {tuned_seconds:.0f}s")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

سه پرسش را با این نتایج جواب دهید:

1. **دقت:** مدل فاین‌تیون‌شده روی الگوهای آشنا تقریباً کامل است. این انتظار می‌رفت، چون دادهٔ آزمون شبیه دادهٔ آموزش است.
2. **تعمیم:** روی **الگوهای جدید** چقدر افت می‌کند؟ این عدد صادقانه‌تر است. افت زیاد یعنی مدل الگوها را حفظ کرده، نه زبان را. راه‌حل: دادهٔ آموزشی متنوع‌تر.
3. **هزینه:** پرامپت کوتاه یعنی توکن ورودی کمتر و پاسخ سریع‌تر. در سرویسی با میلیون‌ها درخواست، همین تفاوت هزینهٔ واقعی دارد.

ارزیابی روی پیام‌های واقعی و انتشار مدل موضوع جلسهٔ بعد است. فعلاً آداپتور LoRA را ذخیره می‌کنیم. آداپتور فقط ماتریس‌های A و B است و حجمش چند ده مگابایت است، نه چند گیگابایت.

</div>

In [ ]:
trainer.model.save_pretrained("order-extractor-lora")
tokenizer.save_pretrained("order-extractor-lora")
!du -sh order-extractor-lora

# Optional: keep the adapter in Google Drive for the next session
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r order-extractor-lora /content/drive/MyDrive/

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۹. جمع‌بندی

- پیش از فاین‌تیون، پرامپت بهتر و RAG را امتحان کنید. فاین‌تیون برای رفتار، قالب و کارهای محدود و پرتکرار است، نه برای دانش متغیر.
- LoRA وزن‌های اصلی را ثابت نگه می‌دارد و یک اصلاحیهٔ کم‌رتبه $BA$ یاد می‌گیرد که کمتر از یک درصد پارامترها را دارد.
- QLoRA مدل پایه را ۴ بیتی نگه می‌دارد تا فاین‌تیون روی GPU کوچک ممکن شود.
- در قالب Prompt-Completion، loss فقط روی پاسخ حساب می‌شود و Chat Template آموزش و استفاده باید یکی باشد.
- ارزیابی روی دادهٔ مشابه آموزش خوش‌بینانه است. الگوهای جدید و دادهٔ واقعی تصویر صادقانه‌تری می‌دهند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. آموزش را با `r=4` و `r=64` تکرار کنید (و `lora_alpha` را دو برابر `r` بگذارید). دقت روی الگوهای جدید، زمان آموزش و حجم آداپتور چه تغییری می‌کند؟
۲. منحنی یادگیری بسازید: مدل را با ۱۰۰، ۳۰۰ و ۱۲۰۰ نمونهٔ آموزشی آموزش دهید و دقت کامل را روی آزمون رسم کنید. چند نمونه برای این کار کافی است؟
۳. مدل فاین‌تیون‌شده را روی `real_data` ارزیابی کنید و خطاها را دسته‌بندی کنید. کدام ویژگی پیام‌های واقعی در دادهٔ مصنوعی نبوده است؟
۴. فرض کنید فروشگاه کالای جدیدی با شناسهٔ `P1007` اضافه می‌کند. مدل فاین‌تیون‌شده با پیامی دربارهٔ این کالا چه می‌کند؟ دو راه برای پشتیبانی از کالاهای جدید پیشنهاد دهید و مزایا و معایب هر کدام را بنویسید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** آموزش را با `r=4` و `r=64` تکرار کنید (و `lora_alpha` را دو برابر `r` بگذارید). دقت روی الگوهای جدید، زمان آموزش و حجم آداپتور چه تغییری می‌کند؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** منحنی یادگیری بسازید: مدل را با ۱۰۰، ۳۰۰ و ۱۲۰۰ نمونهٔ آموزشی آموزش دهید و دقت کامل را روی آزمون رسم کنید. چند نمونه برای این کار کافی است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** مدل فاین‌تیون‌شده را روی `real_data` ارزیابی کنید و خطاها را دسته‌بندی کنید. کدام ویژگی پیام‌های واقعی در دادهٔ مصنوعی نبوده است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** فرض کنید فروشگاه کالای جدیدی با شناسهٔ `P1007` اضافه می‌کند. مدل فاین‌تیون‌شده با پیامی دربارهٔ این کالا چه می‌کند؟ دو راه برای پشتیبانی از کالاهای جدید پیشنهاد دهید و مزایا و معایب هر کدام را بنویسید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید